# GroupDNA — WhatsApp Group Chat Analytics

**Student Name:** Naga Chaitanya    


### Project objective
GroupDNA reads the supplied `hostel_bois.txt` WhatsApp export and creates a personality and activity report using Python fundamentals and NumPy.

**Constraint followed:** No pandas, Counter, regex, matplotlib, seaborn, plotly, or pre-built WhatsApp analyzer.  
**AI-assisted disclosure:** AI was used as a learning aid to help structure/debug parts of this notebook, as permitted by the project brief.


## Setup and data loading

In [ ]:
import numpy as np
from datetime import datetime, timedelta

FILE_PATH = "hostel_bois.txt"

with open(FILE_PATH, "r", encoding="utf-8") as file:
    raw_lines = file.read().splitlines()

print(f"Loaded {len(raw_lines):,} lines from {FILE_PATH}.")


Loaded 3,178 lines from hostel_bois.txt.


## Feature 1 — Chat Parser

In [ ]:
# AI-assisted: I used AI as a learning aid while debugging the WhatsApp line parsing logic.

DATE_LENGTH = 8

def looks_like_date(line):
    """Check the DD/MM/YY prefix without using regex."""
    return (
        len(line) >= DATE_LENGTH
        and line[2] == "/"
        and line[5] == "/"
        and line[:2].isdigit()
        and line[3:5].isdigit()
        and line[6:8].isdigit()
    )

def parse_chat(lines):
    messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0
    continuation_count = 0

    current = None

    for raw in lines:
        line = raw.strip()

        if line == "":
            continue

        # Bonus handling for multi-line messages.
        if not looks_like_date(line):
            if current is not None:
                current["text"] += " " + line
                continuation_count += 1
            continue

        if " - " not in line:
            continue

        timestamp, remainder = line.split(" - ", 1)

        # A sender message must contain ": " after the timestamp.
        if ": " not in remainder:
            system_count += 1
            current = None
            continue

        sender, message_text = remainder.split(": ", 1)

        record = {
            "timestamp": timestamp,
            "sender": sender,
            "text": message_text,
            "dt": datetime.strptime(timestamp, "%d/%m/%y, %H:%M")
        }

        messages.append(record)
        current = record

        if message_text == "<Media omitted>":
            media_count += 1
        elif message_text == "This message was deleted":
            deleted_count += 1

    return messages, system_count, media_count, deleted_count, continuation_count

messages, system_count, media_count, deleted_count, continuation_count = parse_chat(raw_lines)

participants = sorted({message["sender"] for message in messages})
message_dates = [message["dt"].date() for message in messages]

print("=" * 65)
print("PARSER CHECK")
print("=" * 65)
print(f"Activity messages parsed : {len(messages):,}")
print(f"Participants             : {len(participants)}")
print(f"System messages skipped  : {system_count}")
print(f"Media entries            : {media_count}")
print(f"Deleted entries          : {deleted_count}")
print(f"Continuation lines       : {continuation_count}")
print(f"Participants             : {', '.join(participants)}")
print("=" * 65)

print("\\nFirst 5 parsed messages:")
for item in messages[:5]:
    print(item)

print("\\nLast 5 parsed messages:")
for item in messages[-5:]:
    print(item)


PARSER CHECK
Activity messages parsed : 3,174
Participants             : 6
System messages skipped  : 4
Media entries            : 32
Deleted entries          : 15
Continuation lines       : 0
Participants             : Aman, Karan, Neha, Priya, Rahul, Vikas
\nFirst 5 parsed messages:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix', 'dt': datetime.datetime(2024, 4, 1, 1, 17)}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan', 'dt': datetime.datetime(2024, 4, 1, 1, 17)}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene', 'dt': datetime.datetime(2024, 4, 1, 1, 18)}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?', 'dt': datetime.datetime(2024, 4, 1, 2, 13)}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey', 'dt': datetime.datetime(2024, 4, 1, 2, 13)}
\nLast 5 parsed messages:
{'timestamp': '30/05/24, 19:14', 'sender': 'Priya', 'text': 'Take care everyone', 'dt': datetime.datetime(20

## Feature 2 — Group Overview

In [ ]:
def message_count_by_person(records):
    counts = {}
    for record in records:
        person = record["sender"]
        counts[person] = counts.get(person, 0) + 1
    return counts

def special_count_by_person(records, special_text):
    counts = {}
    for record in records:
        if record["text"] == special_text:
            person = record["sender"]
            counts[person] = counts.get(person, 0) + 1
    return counts

def human_date(value):
    return value.strftime("%d %B %Y").lstrip("0")

person_counts = message_count_by_person(messages)
media_by_person = special_count_by_person(messages, "<Media omitted>")
deleted_by_person = special_count_by_person(messages, "This message was deleted")

first_date = min(message_dates)
last_date = max(message_dates)
total_days = (last_date - first_date).days + 1

sorted_counts = sorted(person_counts.items(), key=lambda item: item[1], reverse=True)

print("=" * 65)
print('GROUP OVERVIEW — "Hostel Bois 4ever"')
print("=" * 65)
print(f"Period       : {human_date(first_date)} to {human_date(last_date)} ({total_days} days)")
print(f"Total messages: {len(messages):,}")
print(f"Participants  : {len(participants)}")
print()
print("MESSAGES PER PERSON")
print("-" * 65)

for person, count in sorted_counts:
    percentage = count / len(messages) * 100
    print(f"{person:<10} : {count:>4} ({percentage:5.1f}%)")

print()
print("SPECIAL MESSAGE COUNTS")
for person in participants:
    print(f"{person:<10} : media={media_by_person.get(person, 0):>2} | deleted={deleted_by_person.get(person, 0):>2}")


GROUP OVERVIEW — "Hostel Bois 4ever"
Period       : 1 April 2024 to 30 May 2024 (60 days)
Total messages: 3,174
Participants  : 6

MESSAGES PER PERSON
-----------------------------------------------------------------
Rahul      :  953 ( 30.0%)
Priya      :  718 ( 22.6%)
Neha       :  635 ( 20.0%)
Aman       :  490 ( 15.4%)
Karan      :  354 ( 11.2%)
Vikas      :   24 (  0.8%)

SPECIAL MESSAGE COUNTS
Aman       : media= 4 | deleted= 2
Karan      : media= 7 | deleted= 2
Neha       : media= 8 | deleted= 3
Priya      : media= 4 | deleted= 2
Rahul      : media= 7 | deleted= 6
Vikas      : media= 2 | deleted= 0


## Feature 3 — Most Active Day and Hour

In [ ]:
def count_by_date(records):
    result = {}
    for record in records:
        day = record["dt"].date()
        result[day] = result.get(day, 0) + 1
    return result

def count_by_hour(records):
    result = {}
    for record in records:
        hour = record["dt"].hour
        result[hour] = result.get(hour, 0) + 1
    return result

daily_counts = count_by_date(messages)
hourly_counts = count_by_hour(messages)

busiest_day = max(daily_counts, key=daily_counts.get)
busiest_hour = max(hourly_counts, key=hourly_counts.get)

print("=" * 65)
print("ACTIVITY PEAKS")
print("=" * 65)
print(f"Busiest day  : {human_date(busiest_day)} ({daily_counts[busiest_day]} messages)")
print(f"Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 ({hourly_counts[busiest_hour]} messages)")
print(f"Average per day: {len(messages) / total_days:.1f} messages")


ACTIVITY PEAKS
Busiest day  : 4 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (248 messages)
Average per day: 52.9 messages


## Feature 4 — NumPy Activity Heatmap

In [ ]:
# Rows = participants, columns = hours 00 to 23.
activity_matrix = np.zeros((len(participants), 24), dtype=int)
person_row = {person: index for index, person in enumerate(participants)}

for record in messages:
    row = person_row[record["sender"]]
    hour = record["dt"].hour
    activity_matrix[row, hour] += 1

def heat_symbol(value, row_max):
    if value == 0 or row_max == 0:
        return ". "
    ratio = value / row_max
    if ratio < 0.25:
        return "░ "
    if ratio < 0.50:
        return "▒ "
    if ratio < 0.75:
        return "▓ "
    return "█ "

print("=" * 65)
print("ACTIVITY HEATMAP — messages by hour")
print("=" * 65)
print("Hour  " + " ".join(f"{hour:02d}" for hour in range(24)))

for row_index, person in enumerate(participants):
    row = activity_matrix[row_index]
    row_max = np.max(row)
    symbols = "".join(heat_symbol(value, row_max) for value in row)
    print(f"{person:<7}{symbols}")

print()
print("Row totals checked with NumPy:", np.sum(activity_matrix, axis=1))
print("Total matrix messages:", int(np.sum(activity_matrix)))
print()
print("Night-hour concentration (23:00 to 04:59):")
for row_index, person in enumerate(participants):
    night_total = np.sum(activity_matrix[row_index, [23, 0, 1, 2, 3, 4]])
    share = night_total / np.sum(activity_matrix[row_index]) * 100
    print(f"{person:<10}: {share:5.1f}%")


ACTIVITY HEATMAP — messages by hour
Hour  00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman   ▓ █ █ ▓ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . ▓ 
Karan  . . . . . . . ░ ▒ ▒ ▓ ▒ █ ▓ █ ▓ ▓ ▓ ▓ █ ▓ ▒ ░ ░ 
Neha   . . . . . ▒ ░ ░ ▓ █ █ ▒ ▓ ▓ ▒ ░ ▓ █ █ █ ▓ ▒ ▒ ▒ 
Priya  . . . . . . ░ ▒ ▓ █ █ █ █ ▓ ▓ ▒ ▒ ▓ ▓ █ ▓ ▒ ▒ ░ 
Rahul  ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ▓ ▒ ▒ ▓ ▓ ▒ █ ▓ ▒ █ ▓ ▓ 
Vikas  . . . . . . . ▒ █ ▒ ▒ . ▓ ▓ . ▒ ▒ █ ▓ ▓ ▒ ▒ ▒ ▓ 

Row totals checked with NumPy: [490 354 635 718 953  24]
Total matrix messages: 3174

Night-hour concentration (23:00 to 04:59):
Aman      :  79.8%
Karan     :   2.3%
Neha      :   4.7%
Priya     :   1.3%
Rahul     :  13.4%
Vikas     :   8.3%


## Feature 5 — Top Words

In [ ]:
# Stop words are defined manually so the project does not depend on NLP libraries.
STOP_WORDS = {
    "i", "is", "the", "a", "an", "and", "or", "to", "of", "in", "on", "for",
    "you", "me", "my", "we", "it", "this", "that", "are", "am", "was", "were",
    "be", "with", "at", "as", "do", "did", "have", "has", "had", "but", "if",
    "so", "from", "they", "he", "she", "how", "about", "today", "his", "just",
    "which", "everyone", "telling", "up", "one", "started", "no", "not", "all",
    "more", "very", "been", "being", "get", "got", "going", "go", "come", "came",
    "like", "okay", "yeah", "yes", "now", "everything", "anyone", "entire"
}

PUNCTUATION = ".,!?;:'\"()[]{}<>-/\\\\|@#$%^&*_+=~`"

def clean_word(raw_word):
    word = raw_word.lower().strip(PUNCTUATION)
    return word

word_counts = {}

for record in messages:
    text_value = record["text"]

    if text_value == "<Media omitted>" or text_value == "This message was deleted":
        continue

    for raw_word in text_value.split():
        word = clean_word(raw_word)

        if word == "" or word in STOP_WORDS:
            continue

        word_counts[word] = word_counts.get(word, 0) + 1

top_words = sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:10]
largest_word_count = top_words[0][1] if top_words else 1

print("=" * 65)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 65)

for word, count in top_words:
    bar_length = max(1, int(count / largest_word_count * 24))
    print(f"{word:<12} {'█' * bar_length:<24} {count}")


THIS GROUP'S FAVOURITE WORDS
guys         ████████████████████████ 318
hai          ████████████████████     268
bhai         ████████████             160
scene        ██████████               145
please       ██████████               141
yaar         ██████████               139
kya          ██████████               133
sleep        ████████                 112
who          ████████                 106
what         ███████                  103


## Feature 6 — Response Speed & Silent Streaks

In [ ]:
def average_response_times(records):
    # A response is treated as an active-chat response when the next message
    # from the person follows another person's message within 60 minutes.
    # This avoids calling long periods of group inactivity a "slow response".
    gaps_by_person = {person: [] for person in participants}
    previous = None

    for record in records:
        if previous is not None and record["sender"] != previous["sender"]:
            gap_seconds = (record["dt"] - previous["dt"]).total_seconds()
            if 0 <= gap_seconds <= 60 * 60:
                gaps_by_person[record["sender"]].append(gap_seconds)
        previous = record

    averages = {}
    for person in participants:
        gaps = gaps_by_person[person]
        averages[person] = np.mean(gaps) if gaps else 0

    return averages, gaps_by_person

def longest_silent_streak(records, start_day, end_day, person):
    active_days = {record["dt"].date() for record in records if record["sender"] == person}

    longest = 0
    current = 0
    best_start = None
    best_end = None
    current_start = None

    day = start_day
    while day <= end_day:
        if day not in active_days:
            if current == 0:
                current_start = day
            current += 1

            if current > longest:
                longest = current
                best_start = current_start
                best_end = day
        else:
            current = 0
            current_start = None

        day += timedelta(days=1)

    return longest, best_start, best_end

avg_response, response_gaps = average_response_times(messages)

print("=" * 65)
print("RESPONSE PATTERNS")
print("=" * 65)

fastest = min(avg_response, key=avg_response.get)
slowest = max(avg_response, key=avg_response.get)

def format_duration(seconds):
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f} minutes"
    return f"{minutes / 60:.1f} hours"

print(f"Fastest replier : {fastest} (avg {format_duration(avg_response[fastest])})")
print(f"Slowest replier : {slowest} (avg {format_duration(avg_response[slowest])})")

silent_results = {}
print()
print("LONGEST SILENT STREAKS")
for person in sorted(participants, key=lambda p: longest_silent_streak(messages, first_date, last_date, p)[0], reverse=True):
    silent_results[person] = longest_silent_streak(messages, first_date, last_date, person)
    streak, streak_start, streak_end = silent_results[person]

    if streak == 0:
        print(f"{person:<10}: 0 days (active every day)")
    else:
        print(f"{person:<10}: {streak:>2} days ({human_date(streak_start)} - {human_date(streak_end)})")


RESPONSE PATTERNS
Fastest replier : Rahul (avg 19.3 minutes)
Slowest replier : Vikas (avg 24.6 minutes)

LONGEST SILENT STREAKS
Vikas     : 11 days (23 April 2024 - 3 May 2024)
Aman      : 0 days (active every day)
Karan     : 0 days (active every day)
Neha      : 0 days (active every day)
Priya     : 0 days (active every day)
Rahul     : 0 days (active every day)


## Feature 7 — Personality Archetype Detection

In [ ]:
# Each function returns a score from 0 to 100.
# Primary archetypes use the thresholds from the brief.
# Comedian and Question Master are fallback archetypes with lower maximum scores,
# so a participant who clearly qualifies for a primary archetype keeps that identity.
# The final assignment is exclusive: one archetype per participant.

def person_messages(records, person):
    return [record for record in records if record["sender"] == person]

def burst_average(person):
    runs = []
    run = 0

    for record in messages:
        if record["sender"] == person:
            run += 1
        else:
            if run:
                runs.append(run)
            run = 0

    if run:
        runs.append(run)

    return np.mean(runs) if runs else 0

def spammer_score(person_records):
    average_burst = burst_average(person_records[0]["sender"]) if person_records else 0
    if average_burst > 3:
        return min(100, 70 + (average_burst - 3) / 2 * 30)
    return average_burst / 3 * 35

def caring_count(person_records):
    caring_keywords = [
        "okay", "safe", "eat", "sleep", "take care", "are you",
        "please", "reminder", "drink water", "don't forget"
    ]
    count = 0
    for record in person_records:
        lower_text = record["text"].lower()
        for keyword in caring_keywords:
            count += lower_text.count(keyword)
    return count

def group_mom_score(person_records):
    maximum = max(caring_count(person_messages(messages, person)) for person in participants)
    return caring_count(person_records) / maximum * 100 if maximum else 0

def night_owl_score(person_records):
    if not person_records:
        return 0
    night_count = sum(
        1 for record in person_records
        if record["dt"].hour >= 23 or record["dt"].hour <= 4
    )
    share = night_count / len(person_records)
    if share > 0.60:
        return 100
    return share / 0.60 * 40

def storyteller_score(person_records):
    usable = [
        record for record in person_records
        if record["text"] not in ("<Media omitted>", "This message was deleted")
    ]
    if not usable:
        return 0

    average_words = np.mean([len(record["text"].split()) for record in usable])
    if average_words > 30:
        return 100
    return average_words / 30 * 40

def drama_queen_score(person_records):
    if not person_records:
        return 0

    qualifying = [record for record in person_records if len(record["text"]) >= 3]
    all_caps = sum(1 for record in qualifying if record["text"].isupper())
    exclamation = sum(1 for record in person_records if record["text"].count("!") >= 2)

    caps_share = all_caps / len(qualifying) if qualifying else 0
    exclaim_share = exclamation / len(person_records)

    if caps_share > 0.30 or exclaim_share > 0.30:
        return 100

    return min(40, max(caps_share / 0.30 * 40, exclaim_share / 0.30 * 40))

def ghost_score(person_records):
    person = person_records[0]["sender"] if person_records else ""
    active_days = {record["dt"].date() for record in messages if record["sender"] == person}
    silent_day_count = total_days - len(active_days)
    silent_share = silent_day_count / total_days

    if silent_share > 0.60:
        return 100

    return silent_share / 0.60 * 40

def comedian_score(person_records):
    funny_words = ["lol", "lmao", "haha", "rofl", "lmfao"]
    if not person_records:
        return 0

    funny_count = sum(
        sum(record["text"].lower().count(word) for word in funny_words)
        for record in person_records
    )
    share = funny_count / len(person_records)

    max_share = 0
    for person in participants:
        records = person_messages(messages, person)
        count = sum(
            sum(record["text"].lower().count(word) for word in funny_words)
            for record in records
        )
        if records:
            max_share = max(max_share, count / len(records))

    # Fallback only: maximum 35.
    return share / max_share * 35 if max_share else 0

def question_master_score(person_records):
    if not person_records:
        return 0

    question_count = sum(
        1 for record in person_records if record["text"].rstrip().endswith("?")
    )
    share = question_count / len(person_records)

    # Fallback only: maximum 35.
    if share > 0.25:
        return 35
    return share / 0.25 * 35

archetype_functions = {
    "THE SPAMMER": spammer_score,
    "THE GROUP MOM": group_mom_score,
    "THE NIGHT OWL": night_owl_score,
    "THE STORYTELLER": storyteller_score,
    "THE DRAMA QUEEN": drama_queen_score,
    "THE GHOST": ghost_score,
    "THE COMEDIAN": comedian_score,
    "THE QUESTION MASTER": question_master_score
}

tie_order = [
    "THE SPAMMER", "THE GROUP MOM", "THE NIGHT OWL", "THE STORYTELLER",
    "THE DRAMA QUEEN", "THE GHOST", "THE COMEDIAN", "THE QUESTION MASTER"
]

all_scores = {}
assignments = {}

for person in participants:
    records = person_messages(messages, person)
    scores = {}

    for name in tie_order:
        scores[name] = archetype_functions[name](records)

    all_scores[person] = scores

    ranked = sorted(
        tie_order,
        key=lambda name: (-scores[name], tie_order.index(name))
    )
    assignments[person] = ranked[0]

print("=" * 65)
print("PERSONALITY ARCHETYPES")
print("=" * 65)

for person in sorted(participants, key=lambda p: tie_order.index(assignments[p])):
    archetype = assignments[person]
    score = all_scores[person][archetype]
    print(f"{person:<10} → {archetype:<20} (score {score:5.1f})")

print()
print("ARCHETYPE SCORE TABLE")
print("-" * 65)
for person in participants:
    print(person)
    for name in tie_order:
        print(f"  {name:<22}: {all_scores[person][name]:5.1f}")


PERSONALITY ARCHETYPES
Rahul      → THE SPAMMER          (score  92.7)
Priya      → THE GROUP MOM        (score 100.0)
Aman       → THE NIGHT OWL        (score 100.0)
Karan      → THE STORYTELLER      (score 100.0)
Neha       → THE DRAMA QUEEN      (score 100.0)
Vikas      → THE GHOST            (score 100.0)

ARCHETYPE SCORE TABLE
-----------------------------------------------------------------
Aman
  THE SPAMMER           :  31.8
  THE GROUP MOM         :  15.9
  THE NIGHT OWL         : 100.0
  THE STORYTELLER       :   6.7
  THE DRAMA QUEEN       :   0.0
  THE GHOST             :   0.0
  THE COMEDIAN          :   0.0
  THE QUESTION MASTER   :   9.1
Karan
  THE SPAMMER           :  14.4
  THE GROUP MOM         :   4.7
  THE NIGHT OWL         :   1.5
  THE STORYTELLER       : 100.0
  THE DRAMA QUEEN       :   0.0
  THE GHOST             :   0.0
  THE COMEDIAN          :   0.0
  THE QUESTION MASTER   :   0.0
Neha
  THE SPAMMER           :  30.0
  THE GROUP MOM         :   3.7
  THE NI

## Bonus — Invented Archetype: THE HOSTEL PLANNER

In [ ]:
# Bonus rule:
# A "Hostel Planner" frequently uses practical coordination words such as
# "assignment", "class", "lab", "exam", "notes", "project", "tomorrow".
# This score is reported as a bonus metric but does not replace the required
# eight-archetype exclusive assignment.

planner_words = [
    "assignment", "class", "lab", "exam", "notes",
    "project", "tomorrow", "submission", "college"
]

def hostel_planner_score(person_records):
    if not person_records:
        return 0
    hits = 0
    for record in person_records:
        lower_text = record["text"].lower()
        for word in planner_words:
            hits += lower_text.count(word)
    return hits / len(person_records) * 100

print("HOSTEL PLANNER BONUS SCORES")
for person in participants:
    score = hostel_planner_score(person_messages(messages, person))
    print(f"{person:<10}: {score:6.2f}")


HOSTEL PLANNER BONUS SCORES
Aman      :   1.43
Karan     :  47.18
Neha      :   0.00
Priya     :  12.40
Rahul     :   3.67
Vikas     :   8.33


## Feature 8 — Final GroupDNA Report

In [ ]:
def report_bar(value, maximum, width=20):
    if maximum == 0:
        return ""
    return "█" * max(1, int(value / maximum * width))

print()
print("=" * 72)
print('GROUPDNA REPORT — "Hostel Bois 4ever"')
print("=" * 72)
print(f"{total_days} days • {len(messages):,} messages • {len(participants)} members")
print("-" * 72)
print(f"Period       : {human_date(first_date)} to {human_date(last_date)}")
print(f"Busiest day  : {human_date(busiest_day)} ({daily_counts[busiest_day]} messages)")
print(f"Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00 ({hourly_counts[busiest_hour]} messages)")

print()
print("MESSAGES PER PERSON")
max_person_count = max(person_counts.values())
for person, count in sorted_counts:
    print(f"{person:<10} {report_bar(count, max_person_count)} {count:>4} ({count / len(messages) * 100:4.1f}%)")

print()
print("ACTIVITY HEATMAP")
print("       " + " ".join(f"{hour:02d}" for hour in range(24)))
for row_index, person in enumerate(participants):
    row = activity_matrix[row_index]
    row_max = np.max(row)
    symbols = "".join(heat_symbol(value, row_max) for value in row)
    night_share = np.sum(row[[23, 0, 1, 2, 3, 4]]) / np.sum(row) * 100
    tag = "  <- NIGHT OWL" if night_share > 60 else ""
    print(f"{person:<7}{symbols}{tag}")

print()
print("THIS GROUP'S FAVOURITE WORDS")
for word, count in top_words:
    print(f"{word:<12} {report_bar(count, largest_word_count, 24):<24} {count}")

print()
print("RESPONSE PATTERNS")
print(f"Fastest replier : {fastest} (avg {format_duration(avg_response[fastest])})")
print(f"Slowest replier : {slowest} (avg {format_duration(avg_response[slowest])})")

print()
print("LONGEST SILENT STREAKS")
for person in sorted(participants, key=lambda p: silent_results[p][0], reverse=True):
    streak, streak_start, streak_end = silent_results[person]
    if streak == 0:
        print(f"{person:<10}: 0 days")
    else:
        print(f"{person:<10}: {streak:>2} days ({human_date(streak_start)} - {human_date(streak_end)})")

print()
print("PERSONALITY ARCHETYPES")
for person in sorted(participants, key=lambda p: tie_order.index(assignments[p])):
    print(f"{person:<10} → {assignments[person]}")

print()
print("=" * 72)
print("Generated by GroupDNA • Built with Python + NumPy")
print("=" * 72)



GROUPDNA REPORT — "Hostel Bois 4ever"
60 days • 3,174 messages • 6 members
------------------------------------------------------------------------
Period       : 1 April 2024 to 30 May 2024
Busiest day  : 4 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (248 messages)

MESSAGES PER PERSON
Rahul      ████████████████████  953 (30.0%)
Priya      ███████████████  718 (22.6%)
Neha       █████████████  635 (20.0%)
Aman       ██████████  490 (15.4%)
Karan      ███████  354 (11.2%)
Vikas      █   24 ( 0.8%)

ACTIVITY HEATMAP
       00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman   ▓ █ █ ▓ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . ▓   <- NIGHT OWL
Karan  . . . . . . . ░ ▒ ▒ ▓ ▒ █ ▓ █ ▓ ▓ ▓ ▓ █ ▓ ▒ ░ ░ 
Neha   . . . . . ▒ ░ ░ ▓ █ █ ▒ ▓ ▓ ▒ ░ ▓ █ █ █ ▓ ▒ ▒ ▒ 
Priya  . . . . . . ░ ▒ ▓ █ █ █ █ ▓ ▓ ▒ ▒ ▓ ▓ █ ▓ ▒ ▒ ░ 
Rahul  ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ░ ▓ ▒ ▒ ▓ ▓ ▒ █ ▓ ▒ █ ▓ ▓ 
Vikas  . . . . . . . ▒ █ ▒ ▒ . ▓ ▓ . ▒ ▒ █ ▓ ▓ ▒ ▒ ▒ ▓ 

THIS GROUP'S FAVOURITE WORDS
guys      

## Validation Notes

The supplied dataset is synthetic and contains:
- 3,178 physical lines
- 3,174 sender messages used for activity/message counts
- 32 `<Media omitted>` entries
- 15 `This message was deleted` entries
- 4 system messages
- 6 participants
- 60 calendar days

The notebook deliberately keeps media and deleted entries in message/activity totals, while excluding their unavailable content from word-frequency and message-length calculations. This matches the edge-case instructions in the project brief.


## Reflection

The hardest part of this project was parsing a raw WhatsApp export because not every line represents a normal user message. System messages, media placeholders, deleted messages, and possible continuation lines need different treatment.

If I did the project again, I would separate the parsing, statistics, and reporting layers even more clearly and add more automated validation checks for each feature.

The NumPy heatmap was useful because it converts the raw timestamps into a compact 6 × 24 activity matrix. The archetype section then uses those statistics and other message-level features to assign one exclusive personality archetype to each participant.

For the submission, the notebook is designed to run from top to bottom when `hostel_bois.txt` is in the same folder as the notebook.
